In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# This notebook is a copy of "statistics". Its purpose is to run the statistics of a defined number of rows of the train dataset. Such that the statistics can be computed in parallel

In [1]:
import polars as pl
import time
import psutil
import os
import numpy as np
from tqdm.notebook import tqdm  # notebook-friendly progress bar

def get_statistics(data_in, cols_in, n_rows_total, batch_size=500_000,row_offset=0):
    '''
    Computes mean, std, min, max over cols_in with a progress bar.
    Splits the data into batches of batch_size rows and aggregates.
    '''
    n_batches = int(np.ceil(n_rows_total / batch_size))
    
    # Accumulators for online statistics
    batch_means = []
    batch_stds  = []
    batch_mins  = []
    batch_maxs  = []
    batch_counts = []

    source = data_in.lazy().select(cols_in)
    if row_offset > 0:
        source = source.slice(row_offset,n_rows_total)    # skip first half, then read sequentially

    batches = source.collect_batches(chunk_size=batch_size)

    with tqdm(total=n_batches, desc="Computing statistics", unit="batch") as pbar:
        # for i in range(n_batches):
        for batch_df in batches:
            n = len(batch_df)
            if n == 0:
                continue
            #start = row_offset + i * batch_size
            
            # stats = (
            #     data_in.lazy()
            #     .select(cols_in)
            #     .slice(start, batch_size)          # explicit row window
            #     .select([
            #         pl.all().mean().name.suffix("_mean"),
            #         pl.all().std() .name.suffix("_std"),
            #         pl.all().min() .name.suffix("_min"),
            #         pl.all().max() .name.suffix("_max"),
            #         pl.lit(batch_size).alias("_count")
            #     ])
            #     .collect(engine="streaming")
            # )
            stats = (batch_df.select([
                    pl.all().mean().name.suffix("_mean"),
                    pl.all().std() .name.suffix("_std"),
                    pl.all().min() .name.suffix("_min"),
                    pl.all().max() .name.suffix("_max"),
                    pl.lit(n).alias("_count")
                ])
            )
            
            batch_means .append([stats[f"{col}_mean"][0] for col in cols_in])
            batch_stds  .append([stats[f"{col}_std"][0]  for col in cols_in])
            batch_mins  .append([stats[f"{col}_min"][0]  for col in cols_in])
            batch_maxs  .append([stats[f"{col}_max"][0]  for col in cols_in])
            batch_counts.append(stats["_count"][0])

            # pbar.set_postfix({
            #     "rows processed": f"{min((i+1)*batch_size, n_rows_total):,}",
            #     "remaining"     : f"{max(n_rows_total - (i+1)*batch_size, 0):,}"
            # })
            pbar.set_postfix({
                "rows": f"{sum(batch_counts):,}",
                "RAM" : f"{psutil.Process().memory_info().rss/1e9:.1f} GB"
            })
            pbar.update(1)

    # --- Combine batches ---
    # min/max: just take element-wise min/max across batches
    min_out = np.min (batch_mins,  axis=0).tolist()
    max_out = np.max (batch_maxs,  axis=0).tolist()

    # mean: weighted average across batches
    counts  = np.array(batch_counts)             # shape (n_batches,)
    means   = np.array(batch_means)              # shape (n_batches, n_cols)
    mean_out = (means * counts[:, None]).sum(axis=0) / counts.sum()

    # std: combine via sum of squares (correct global std)
    stds    = np.array(batch_stds)               # shape (n_batches, n_cols)
    # variance = mean of (var_i + mean_i^2) - global_mean^2
    vars_   = stds**2 + means**2                 # E[X^2] per batch
    global_ex2 = (vars_ * counts[:, None]).sum(axis=0) / counts.sum()
    std_out  = np.sqrt(np.clip(global_ex2 - mean_out**2, 0, None)).tolist()

    return mean_out, std_out, min_out, max_out

#---------------------------------------------------------------------------------

def get_statistics_vertical_var(data_in, cols_in, n_rows_total, batch_size=500_000,row_offset=0):
    '''
    Given the structure of the data, each vertical variable is splitted into
    sub-variables for each vertical level. For example state_01 has 60 vertical
    levels. In the dataframe 60 columns will be found with the names:
    state_01_0, state_01_1,..., state_01_59. This function collapses all these
    columns into one to get the mean value per variable in location (lat,lon) and 
    vertical level. For example if "data_in" shape is: (10,540), 540 columns are 
    the product of: "number_var_vertical"*60, so doing 540/60 will give the
    number of variables with vertical dimension ("number_var_vertical"). That
    part corresponds to: np.shape(train_ds_dummy.select(X_col_series))[1]//60).
    Then the other dimension is 60 as each variable has 60 vertical levels. The
    final dimension will be (10,9,60).
    
    Output:
    Statistic for each variable with vertical level. The statistics are perform
    over all vertical levels at the same time.
    '''
    n_levels = 60
    n_vars   = len(cols_in) // n_levels

    var_groups = {
        f"var_{i}": cols_in[i*n_levels:(i+1)*n_levels]
        for i in range(n_vars)
    }

    # Accumulators — one list per variable
    all_means  = [[] for _ in range(n_vars)]
    all_stds   = [[] for _ in range(n_vars)]
    all_mins   = [[] for _ in range(n_vars)]
    all_maxs   = [[] for _ in range(n_vars)]
    all_counts = [[] for _ in range(n_vars)]

    # Single pass — read ALL columns at once, process per variable inside loop
    source  = (
        data_in.lazy()
        .select(cols_in)               # all columns in one read
        .slice(row_offset, n_rows_total)
    )
    batches = source.collect_batches(chunk_size=batch_size)

    with tqdm(total=n_rows_total//batch_size + 1, desc="Reading CSV", unit="batch") as pbar:
        for batch_df in batches:
            if len(batch_df) == 0:
                continue

            # Process all variables on this batch while it's in RAM
            for i, (var_name, level_cols) in enumerate(var_groups.items()):
                stats = (
                    batch_df.select(level_cols)   # slice columns in RAM — free
                    .unpivot(on=level_cols, variable_name="level", value_name="value")
                    .select([
                        pl.col("value").mean().alias("mean"),
                        pl.col("value").std() .alias("std"),
                        pl.col("value").min() .alias("min"),
                        pl.col("value").max() .alias("max"),
                    ])
                )
                all_means [i].append(stats["mean"][0])
                all_stds  [i].append(stats["std"][0])
                all_mins  [i].append(stats["min"][0])
                all_maxs  [i].append(stats["max"][0])
                all_counts[i].append(len(batch_df) * n_levels)

            pbar.update(1)

    # Combine batches per variable
    mean_out, std_out, min_out, max_out = [], [], [], []

    for i in range(n_vars):
        counts = np.array(all_counts[i])
        means  = np.array(all_means[i])
        stds   = np.array(all_stds[i])

        m = (means * counts).sum() / counts.sum()
        ex2 = stds**2 + means**2
        s = np.sqrt(np.clip((ex2 * counts).sum() / counts.sum() - m**2, 0, None))

        mean_out.append(m)
        std_out .append(s)
        min_out .append(np.min(all_mins[i]))
        max_out .append(np.max(all_maxs[i]))

    return mean_out, std_out, min_out, max_out
    
    

In [2]:
def vertical_NO_vertical(cols_name_in,col_patterns):
    '''
    This function separates columns names into two categories, those with vertical dimensions and those without vertical
    dimension (scalar)
    Inputs:
    cols_name_in: list, total number column names
    col_patterns: list,pattern with the start name of the vertical variables
    Ouptus:
    X_col_series_out: list, columns names with vertical levels 
    X_col_series_NOT_out: list, columns names without vertical levels
    X_col_series_indices_out: array, columns indices with variables that have vertical levels 
    X_col_series_NOT_indices_out: array, columns indices with variables that DON'T have vertical levels 
    '''
    X_col_series_out = [x for x in cols_name_in if np.mean([x.startswith(y) for y in col_patterns])>0] #it will be true
                # only for when the column starts with one of the col_name_parent and the mean will be larger than 0

    X_col_series_NOT_out  = [x for  x in cols_name_in if x not in X_col_series_out ] #Select the columns that don't have vertical dimension
    
    #Checks if the number of columns with and without vertical levels is the same as the initial (combined) number of columns
    print(f"Sum of vertical and not vertical variables equal to initial number of columns: --> {len(X_col_series_out )+len(X_col_series_NOT_out ) == len(cols_name_in)}")
    
    #----------- Obtaining the inidices of the columns for each varaible clase ------------------
    #---------- variables with vertical dimensions
    X_col_series_indices_out = np.asarray([np.where(np.asarray(cols_name_in) == x)[0][0] for x in X_col_series_out ])
    #---------- variables withOUT vertical dimensions
    X_col_series_NOT_indices_out = np.asarray([np.where(np.asarray(cols_name_in) == x)[0][0] for x in X_col_series_NOT_out ])

    return(X_col_series_out, X_col_series_NOT_out, X_col_series_indices_out, X_col_series_NOT_indices_out)

In [3]:
train_df = pl.scan_csv('/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/train.csv',infer_schema_length=1000)

test_df = pl.scan_csv('/kaggle/input/competitions/leap-atmospheric-physics-ai-climsim/test.csv',infer_schema_length=1000)

In [4]:
FEAT_COLS = train_df.collect_schema().names()[1:557] #Input features (columns names)
TARGET_COLS = train_df.collect_schema().names()[557:] #Targets (columns names)
##
#Selecting the variables that have vertical levels
col_name_parent = ['state_t', 'state_q0001', 'state_q0002', 
                   'state_q0003', 'state_u', 'state_v',
                   'pbuf_ozone', 'pbuf_CH4', 'pbuf_N2O'] # These are the basename of the variables with vertical levels
## Obtaining feature columns names and indices for variables with vertical and not vertical levels
(X_col_series,
    X_col_series_NOT,
    X_col_series_indices,
    X_col_series_NOT_indices,
) = vertical_NO_vertical(cols_name_in = FEAT_COLS, col_patterns = col_name_parent)

## ---------- Dividing target variables into variables with vertical leves and variables without vertical levels
y_name_parent = ['ptend_t', 'ptend_q0001', 'ptend_q0002', 'ptend_q0003', 'ptend_u', 'ptend_v']
(y_col_series,
    y_col_series_NOT,
    y_col_series_indices,
    y_col_series_NOT_indices,
) = vertical_NO_vertical(cols_name_in = TARGET_COLS, col_patterns = y_name_parent)

Sum of vertical and not vertical variables equal to initial number of columns: --> True
Sum of vertical and not vertical variables equal to initial number of columns: --> True


In [5]:
#------------- Global statistics for variables with vertirtical levels ------------
#------ at each level and variables without vertical levels --------------
#---------- for all rows: space/time coordinates -------------------------------
#----- NOTE: This is only done in the training dataset!
pl.Config.set_streaming_chunk_size(100_000)

SLICE_START = 2_522_880*1
SLICE_SIZE  = 2_522_880  # second quarter

#-- X: features. input variables
t0 = time.perf_counter()
mean_X, std_X, min_X, max_X = get_statistics(data_in=train_df,cols_in=FEAT_COLS, 
                                             n_rows_total=SLICE_SIZE  , 
                                             batch_size=100_000,
                                             row_offset=SLICE_START)
elapsed = time.perf_counter() - t0
print("*"*20, f" DONE, elapsed time: {elapsed} ","*"*20)

#--Y: target variables. Variables I want to predict
t0 = time.perf_counter()
mean_Y, std_Y, min_Y, max_Y = get_statistics(data_in=train_df,cols_in=TARGET_COLS, 
                                             n_rows_total=SLICE_SIZE  , 
                                             batch_size=100_000,
                                             row_offset=SLICE_START)
elapsed = time.perf_counter() - t0
print("*"*20, f" DONE, elapsed time: {elapsed} ","*"*20)

Computing statistics:   0%|          | 0/26 [00:00<?, ?batch/s]

********************  DONE, elapsed time: 1652.590996034  ********************


Computing statistics:   0%|          | 0/26 [00:00<?, ?batch/s]

********************  DONE, elapsed time: 1421.6630520779997  ********************


In [6]:
#---------------------- Saving statistics ---------------------------
#-- X: features. input variables
np.save('/kaggle/working/mean_X_part1.npy', mean_X)
np.save('/kaggle/working/std_X_part1.npy',  std_X)
np.save('/kaggle/working/min_X_part1.npy',  min_X)
np.save('/kaggle/working/max_X_part1.npy',  max_X)

#--Y: target variables. Variables I want to predict
np.save('/kaggle/working/mean_Y_part1.npy', mean_Y)
np.save('/kaggle/working/std_Y_part1.npy',  std_Y)
np.save('/kaggle/working/min_Y_part1.npy',  min_Y)
np.save('/kaggle/working/max_Y_part1.npy',  max_Y)

In [7]:
!ls

max_X_part1.npy  mean_X_part1.npy  min_X_part1.npy  std_X_part1.npy
max_Y_part1.npy  mean_Y_part1.npy  min_Y_part1.npy  std_Y_part1.npy


In [8]:
#---------- Computing statistcs for vertical variables over the whole -------------
#-------- vertical profile.
SLICE_START = 2_522_880*1
SLICE_SIZE  = 2_522_880  # first quarter

#-- X: features. input variables
mean_X_vertical, std_X_vertical, min_X_vertical, max_X_vertical = get_statistics_vertical_var(data_in=train_df,
                                                 cols_in=X_col_series,
                                                 n_rows_total=SLICE_SIZE, 
                                                 batch_size=100_000,
                                                 row_offset=SLICE_START ) 
#--Y: target variables. Variables I want to predict
mean_Y_vertical, std_Y_vertical, min_Y_vertical, max_Y_vertical = get_statistics_vertical_var(data_in=train_df,
                                                 cols_in=y_col_series,
                                                 n_rows_total=SLICE_SIZE, 
                                                 batch_size=100_000,
                                                 row_offset=SLICE_START) 

Reading CSV:   0%|          | 0/26 [00:00<?, ?batch/s]

Reading CSV:   0%|          | 0/26 [00:00<?, ?batch/s]

In [11]:
#---------------------- Saving statistics ---------------------------
#-- X: features. input variables
np.save('/kaggle/working/mean_X_vertical_part1.npy', mean_X_vertical)
np.save('/kaggle/working/std_X_vertical_part1.npy',  std_X_vertical)
np.save('/kaggle/working/min_X_vertical_part1.npy',  min_X_vertical)
np.save('/kaggle/working/max_X_vertical_part1.npy',  max_X_vertical)

#--Y: target variables. Variables I want to predict
np.save('/kaggle/working/mean_Y_vertical_part1.npy', mean_Y_vertical)
np.save('/kaggle/working/std_Y_vertical_part1.npy',  std_Y_vertical)
np.save('/kaggle/working/min_Y_vertical_part1.npy',  min_Y_vertical)
np.save('/kaggle/working/max_Y_vertical_part1.npy',  max_Y_vertical)

In [13]:
print(f"mean_X_vertical {np.shape(mean_X_vertical)}")
print(f"std_X_vertical {np.shape(std_X_vertical)}")
print(f"min_X_vertical {np.shape(min_X_vertical)}")
print(f"max_X_vertical {np.shape(max_X_vertical)}")
print(f"mean_Y_vertical {np.shape(mean_Y_vertical)}")
print(f"min_Y_vertical {np.shape(min_Y_vertical)}")
print(f"max_Y_vertical {np.shape(max_Y_vertical)}")

mean_X_vertical (9,)
std_X_vertical (9,)
min_X_vertical (9,)
max_X_vertical (9,)
mean_Y_vertical (6,)
min_Y_vertical (6,)
max_Y_vertical (6,)


In [14]:
!ls

max_X_part1.npy		   mean_Y_part1.npy	      std_X_part1.npy
max_X_vertical_part1.npy   mean_Y_vertical_part1.npy  std_X_vertical_part1.npy
max_Y_part1.npy		   min_X_part1.npy	      std_Y_part1.npy
max_Y_vertical_part1.npy   min_X_vertical_part1.npy   std_Y_vertical_part1.npy
mean_X_part1.npy	   min_Y_part1.npy
mean_X_vertical_part1.npy  min_Y_vertical_part1.npy


---

**End of Notebook**